# Export to ONNX — for the in-browser demo

Converts the trained **sub-center ArcFace @384** model into an `.onnx` file that runs in a web
page (no server, no Python). Also produces a quantized **INT8** copy and — importantly —
**checks numerically that the ONNX output matches PyTorch** before you trust it.

What comes out:

| File | Size (approx) | Use |
|---|--:|---|
| `car_model_fp32.onnx` | ~110 MB | guaranteed-correct reference |
| `car_model_int8.onnx` | ~30 MB | what you ship to the browser (if agreement is high) |
| `demo_config.json` | tiny | classes, temperature, threshold for the page |

**Prerequisite:** `modelgate_v2_artifacts.zip` and `dataset_final.zip` in Drive `CapstoneCars/`.
A GPU is *not* required — this runs fine on CPU.

In [ ]:
# ── Cell 1 · Setup ──────────────────────────────────────────────
!pip install -q timm onnx onnxruntime
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, json, glob, os
from pathlib import Path
import timm
dev = 'cpu'          # export on CPU — avoids device-specific ops in the graph
print("torch", torch.__version__, "| timm", timm.__version__)

In [ ]:
# ── Cell 2 · Model defs (verbatim from model_gate_v2) + reload ──
class SubCenterArcFace(nn.Module):
    def __init__(self, in_features, num_classes, K=3, s=30.0, m=0.30):
        super().__init__()
        self.num_classes, self.K, self.s, self.m = num_classes, K, s, m
        self.W = nn.Parameter(torch.empty(num_classes*K, in_features))
        nn.init.xavier_uniform_(self.W)
    def forward(self, feat, labels=None):
        f = F.normalize(feat, dim=1)
        w = F.normalize(self.W, dim=1)
        cos = (f @ w.t()).view(-1, self.num_classes, self.K).amax(dim=2)
        cos = cos.float().clamp(-1+1e-6, 1-1e-6)
        if labels is None:
            return self.s * cos
        theta  = torch.acos(cos)
        margin = torch.zeros_like(cos).scatter_(1, labels.view(-1,1), self.m)
        return self.s * torch.cos(theta + margin)

class ArcModel(nn.Module):
    def __init__(self, model_id, num_classes, K=3, s=30.0, m=0.30, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_id, pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        self.head = SubCenterArcFace(self.backbone.num_features, num_classes, K, s, m)
    def forward(self, x, labels=None):
        return self.head(self.backbone(x), labels)

class SnapMixNet(nn.Module):
    def __init__(self, model_id, num_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_id, pretrained=pretrained,
                                          num_classes=0, global_pool='')
        self.fc = nn.Linear(self.backbone.num_features, num_classes)
    def forward(self, x):
        f = self.backbone.forward_features(x)
        return self.fc(F.adaptive_avg_pool2d(f, 1).flatten(1))

from google.colab import drive
drive.mount('/content/drive')
arts = (glob.glob('/content/drive/MyDrive/**/modelgate_v2_artifacts.zip', recursive=True)
        or glob.glob('/content/drive/MyDrive/**/modelgate_artifacts.zip', recursive=True))
print("artifacts:", arts[0])
!rm -rf /content/artifacts && unzip -o -q "{arts[0]}" -d /content

cfg = json.load(open('/content/artifacts/config.json'))
CLASSES = cfg['classes']; IMG = cfg['img_size']
print("winner:", cfg['winner'], "| head:", cfg.get('head'), "| img:", IMG)
print("classes:", CLASSES)

h = cfg.get('head','linear')
if   h == 'arcface': model = ArcModel(cfg['model_id'], len(CLASSES), pretrained=False, **cfg['arc'])
elif h == 'snapmix': model = SnapMixNet(cfg['model_id'], len(CLASSES), pretrained=False)
else:                model = timm.create_model(cfg['model_id'], pretrained=False, num_classes=len(CLASSES))
model.load_state_dict(torch.load('/content/artifacts/model.pt', map_location='cpu'))
model.eval().to(dev)
print("model reloaded ✓")

In [ ]:
# ── Cell 3 · Export to ONNX (fp32) ──────────────────────────────
# Thin wrapper so the traced graph only ever takes the image (no `labels` branch).
class InferenceWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x)          # ArcFace: labels=None -> s * cos

wrapped = InferenceWrapper(model).eval()
dummy = torch.randn(1, 3, IMG, IMG)

FP32 = '/content/car_model_fp32.onnx'
torch.onnx.export(
    wrapped, dummy, FP32,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,            # LayerNormalization is native from 17; well supported by ORT Web
    do_constant_folding=True,
)
import onnx
onnx.checker.check_model(onnx.load(FP32))
print(f"✅ fp32 exported: {os.path.getsize(FP32)/1e6:.1f} MB")

In [ ]:
# ── Cell 4 · Quantize to INT8 (what the browser downloads) ──────
from onnxruntime.quantization import quantize_dynamic, QuantType
INT8 = '/content/car_model_int8.onnx'
try:
    quantize_dynamic(FP32, INT8, weight_type=QuantType.QUInt8)
    print(f"✅ int8 exported: {os.path.getsize(INT8)/1e6:.1f} MB "
          f"({os.path.getsize(FP32)/os.path.getsize(INT8):.1f}x smaller)")
except Exception as e:
    INT8 = None
    print("⚠️ INT8 quantization failed — ship fp32 instead. Reason:", str(e)[:200])

In [ ]:
# ── Cell 5 · VERIFY: does ONNX match PyTorch? (do not skip) ─────
# Runs both on real test images and compares. If agreement is not ~100%, do NOT ship that file.
import onnxruntime as ort
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

ds = (glob.glob('/content/drive/MyDrive/**/dataset_final.zip', recursive=True)
      or glob.glob('/content/drive/MyDrive/**/dataset_split.zip', recursive=True))[0]
!rm -rf /content/dataset && unzip -o -q "{ds}" -d /content

MEAN, STD = tuple(cfg['mean']), tuple(cfg['std'])
RESIZE = 438 if IMG == 384 else int(round(IMG / 0.875))  # MUST equal model_gate_v2's Resize(438)
eval_tf = transforms.Compose([transforms.Resize(RESIZE), transforms.CenterCrop(IMG),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_ds = datasets.ImageFolder('/content/dataset/test', eval_tf)
assert test_ds.classes == CLASSES, "class order mismatch!"

N = 64
idx = list(range(0, len(test_ds), max(1, len(test_ds)//N)))[:N]
xs = torch.stack([test_ds[i][0] for i in idx])
with torch.no_grad():
    ref = wrapped(xs).numpy()

def check(path, name):
    sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    out = sess.run(['logits'], {'input': xs.numpy()})[0]
    agree = (out.argmax(1) == ref.argmax(1)).mean()
    print(f"{name:6s} | top-1 agreement vs PyTorch: {agree:.1%} | "
          f"max |Δlogit|: {np.abs(out-ref).max():.4f}")
    return agree

a32 = check(FP32, 'fp32')
a8  = check(INT8, 'int8') if INT8 else 0.0
print()
print("fp32 must be ~100%. If int8 is >=98% it is safe to ship; otherwise ship fp32.")
SHIP = INT8 if (INT8 and a8 >= 0.98) else FP32
print("→ ship:", SHIP, f"({os.path.getsize(SHIP)/1e6:.1f} MB)")

In [ ]:
# ── Cell 6 · Write demo_config.json + save everything to Drive ──
demo_cfg = {
    'classes': CLASSES,
    'img_size': IMG,
    'resize': RESIZE,
    'mean': list(MEAN), 'std': list(STD),
    # From the Trust Layer. Until you run it: T=1.0 and no abstention.
    'temperature': float(cfg.get('temperature', 1.0)),
    'abstain_threshold': cfg.get('abstain_threshold', None),
    'reject_class': 'others' if 'others' in CLASSES else None,
    'model_file': os.path.basename(SHIP),
}
json.dump(demo_cfg, open('/content/demo_config.json','w'), indent=2)
print(json.dumps(demo_cfg, indent=2))

!cp "{SHIP}" /content/drive/MyDrive/CapstoneCars/
!cp /content/demo_config.json /content/drive/MyDrive/CapstoneCars/
print("\n✅ copied to Drive/CapstoneCars/:", os.path.basename(SHIP), "+ demo_config.json")
print("\nNext: download it, then either")
print("  A) open docs/demo.html and pick the .onnx with 'Load from your computer', or")
print("  B) upload the .onnx to a Hugging Face model repo and paste its resolve URL.")